# Tái thực nghiệm dự báo giá đồng bằng LSTM

Notebook này tái thực nghiệm theo Chen và cộng sự (2023), PLOS ONE 18(10): e0285631, với khoảng dữ liệu 20 năm kết thúc vào ngày 26/09/2026. Mô hình sử dụng là `LSTM` gồm hai lớp hồi quy.

Paper sử dụng dữ liệu từ Investing.com và lựa chọn giá dầu WTI, vàng và bạc làm biến đầu vào dựa trên tương quan Spearman. Để notebook dễ chạy trên Kaggle/Google Colab, phần tải dữ liệu mặc định dùng `yfinance` làm nguồn thay thế tương thích và lưu lại một file CSV chuẩn.

Mục tiêu của notebook: dự báo giá đóng cửa của đồng, sau đó đánh giá bằng MSE, RMSE và MAE. Notebook SimpleRNN đi kèm dùng cùng dữ liệu và cách chia tập để so sánh công bằng.

## 1. Cài đặt và nạp thư viện

Nếu chạy trên Kaggle/Colab mới, bỏ dấu `#` ở dòng `pip install` nếu môi trường báo thiếu thư viện, sau đó chạy cell này một lần.

In [ ]:
# Nếu thiếu thư viện, bỏ dấu # ở dòng bên dưới rồi chạy một lần.
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow

from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Phiên bản TensorFlow:', tf.__version__)

## 2. Cấu hình thí nghiệm

`START_DATE` và `REFERENCE_END` là khoảng thời gian dùng để lọc dữ liệu; ngày kết thúc được tính bao gồm cả ngày 26/09/2026. Hàm tải dữ liệu của Yahoo Finance dùng ngày kết thúc không bao gồm, nên code tự cộng thêm một ngày vào `DOWNLOAD_END`.

Các biến quan trọng:
- `LOOKBACK = 22`: mỗi mẫu nhìn lại 22 phiên trước.
- `TEST_HORIZON = 242`: giữ lại 242 phiên cuối để kiểm tra; paper cũng dùng 363 và 485.
- `BATCH_SIZE = 64`, `EPOCHS = 100`, `LEARNING_RATE = 5e-5`: thông số gần theo paper.
- `USE_EARLY_STOPPING = False`: giữ đủ 100 epoch để gần cách chạy paper; có thể đổi thành `True` để chạy demo nhanh hơn.
- `USE_LOCAL_CSV = True`: chỉ bật khi bạn có CSV đã xuất từ Investing.com và đã đổi về đúng tên cột chuẩn.

In [ ]:
REFERENCE_END = pd.Timestamp('2026-09-26')
START_DATE = pd.Timestamp('2006-01-01')

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / 'copper_paper_2006_2026.csv'
LOCAL_CSV_PATH = DATA_DIR / 'copper_investing_2006_2026.csv'
KAGGLE_INPUT_DIR = Path('/kaggle/input')
if KAGGLE_INPUT_DIR.exists():
    kaggle_matches = list(KAGGLE_INPUT_DIR.rglob('copper_investing_2006_2026.csv'))
    if kaggle_matches:
        LOCAL_CSV_PATH = kaggle_matches[0]
USE_LOCAL_CSV = True

TARGET = 'closed_copper_price'
SELECTED_FEATURES = ['close_wti_oil', 'close_gold', 'close_silver']
PAPER_COLUMNS = [
    'closed_copper_price', 'volume_copper', 'close_dollar_index',
    'close_t_bill', 'close_Nasdaq', 'close_Dow_Jones_Index',
    'close_wti_oil', 'close_gold', 'close_silver'
]

TICKERS = {
    'copper': 'HG=F',
    'wti': 'CL=F',
    'gold': 'GC=F',
    'silver': 'SI=F',
    'dollar_index': 'DX-Y.NYB',
    't_bill': '^TNX',
    'nasdaq': '^IXIC',
    'dow_jones': '^DJI',
}

LOOKBACK = 22
TEST_HORIZON = 242  # paper cũng báo cáo thêm 363 và 485
VALIDATION_RATIO = 0.10
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 5e-5
DROPOUT = 0.20
USE_EARLY_STOPPING = False  # đổi thành True nếu muốn demo nhanh hơn

print(f'Window: {START_DATE.date()} to {REFERENCE_END.date()}')
print('Selected inputs:', SELECTED_FEATURES)

## 3. Tải và căn chỉnh 9 biến theo paper

Paper mô tả 9 chuỗi dữ liệu: giá đóng cửa đồng, khối lượng đồng, Dollar Index, lợi suất trái phiếu 10 năm, Nasdaq, Dow Jones, dầu WTI, vàng và bạc. Các chuỗi được ghép theo ngày. Nếu một ngày thiếu ít nhất một biến, cả dòng ngày đó sẽ bị loại, giống quy tắc của paper.

Sau khi chạy cell, dữ liệu sạch được lưu tại `data/copper_paper_2006_2026.csv`. Nếu dùng Kaggle, file thường nằm trong `/kaggle/working/data/`.

In [ ]:
def _field_series(raw, field):
    if isinstance(raw.columns, pd.MultiIndex):
        if field in raw.columns.get_level_values(0):
            series = raw.xs(field, axis=1, level=0)
        elif field in raw.columns.get_level_values(1):
            series = raw.xs(field, axis=1, level=1)
        else:
            raise KeyError(f'{field} not found in columns: {raw.columns}')
        if isinstance(series, pd.DataFrame):
            series = series.iloc[:, 0]
    else:
        series = raw[field]
    series = series.copy()
    series.index = pd.to_datetime(series.index).tz_localize(None)
    return series.astype('float64')

def load_paper_dataset():
    if USE_LOCAL_CSV:
        if not LOCAL_CSV_PATH.exists():
            raise FileNotFoundError(f'Missing local CSV: {LOCAL_CSV_PATH}')
        df = pd.read_csv(LOCAL_CSV_PATH, parse_dates=['date']).set_index('date')
        missing = sorted(set(PAPER_COLUMNS) - set(df.columns))
        if missing:
            raise ValueError(f'Local CSV is missing canonical columns: {missing}')
        df = df[PAPER_COLUMNS]
    else:
        raw = {name: _download_one(ticker) for name, ticker in TICKERS.items()}
        series = {
            'closed_copper_price': _field_series(raw['copper'], 'Close'),
            'volume_copper': _field_series(raw['copper'], 'Volume'),
            'close_dollar_index': _field_series(raw['dollar_index'], 'Close'),
            'close_t_bill': _field_series(raw['t_bill'], 'Close'),
            'close_Nasdaq': _field_series(raw['nasdaq'], 'Close'),
            'close_Dow_Jones_Index': _field_series(raw['dow_jones'], 'Close'),
            'close_wti_oil': _field_series(raw['wti'], 'Close'),
            'close_gold': _field_series(raw['gold'], 'Close'),
            'close_silver': _field_series(raw['silver'], 'Close'),
        }
        df = pd.concat(series, axis=1)

    df.index = pd.to_datetime(df.index).tz_localize(None)
    df = df.sort_index()
    df = df.loc[(df.index >= START_DATE) & (df.index <= REFERENCE_END), PAPER_COLUMNS]
    df = df.apply(pd.to_numeric, errors='coerce').dropna()
    if len(df) < TEST_HORIZON + LOOKBACK + 100:
        raise ValueError(f'Only {len(df)} clean rows remain; check the source data.')
    df.to_csv(DATA_PATH, index_label='date')
    return df

df = load_paper_dataset()
print('Số dòng x số cột:', df.shape)
print('Khoảng ngày thực tế:', df.index.min().date(), 'đến', df.index.max().date())
display(df.head())

## 4. Kiểm tra tương quan Spearman

Paper cho biết dầu WTI, vàng và bạc có tương quan cao với giá đồng. Cell này tính lại tương quan Spearman trên khoảng dữ liệu mới 2006-2026.

Bảng kết quả giúp bạn kiểm tra xem 3 biến đầu vào được chọn có còn phù hợp hay không. Heatmap càng gần màu đỏ hoặc xanh đậm thì mức tương quan càng mạnh theo chiều dương hoặc âm.

In [ ]:
spearman = (
    df[PAPER_COLUMNS]
    .corr(method='spearman')[TARGET]
    .sort_values(ascending=False)
)
display(spearman.to_frame('spearman_rho'))

plt.figure(figsize=(9, 6))
sns.heatmap(df[PAPER_COLUMNS].corr(method='spearman'), cmap='coolwarm', center=0, annot=False)
plt.title('Spearman correlation - refreshed window')
plt.tight_layout()
plt.show()

## 5. Tạo chuỗi dữ liệu theo đúng thứ tự thời gian

Biến cần dự báo là `closed_copper_price`. Mỗi mẫu đầu vào gồm 22 phiên gần nhất của dầu WTI, vàng và bạc.

Dữ liệu được chia theo thời gian, không xáo trộn: phần cuối cùng gồm `TEST_HORIZON` phiên là tập test; 10% cuối của phần trước test là tập validation; phần còn lại là tập train. Bộ chuẩn hóa `MinMaxScaler` chỉ học trên train để tránh rò rỉ thông tin từ tương lai.

In [ ]:
def make_sequences(feature_values, target_values, dates, lookback):
    X, y, y_dates = [], [], []
    for end in range(lookback, len(dates)):
        X.append(feature_values[end - lookback:end])
        y.append(target_values[end])
        y_dates.append(dates[end])
    return np.asarray(X, dtype='float32'), np.asarray(y, dtype='float32'), pd.DatetimeIndex(y_dates)

def prepare_split(data, test_horizon):
    test_start = data.index[-test_horizon]
    pretest = data.loc[data.index < test_start]
    val_size = max(1, int(len(pretest) * VALIDATION_RATIO))
    val_start = pretest.index[-val_size]
    train = data.loc[data.index < val_start]

    feature_scaler = MinMaxScaler()
    target_scaler = MinMaxScaler()
    feature_scaler.fit(train[SELECTED_FEATURES])
    target_scaler.fit(train[[TARGET]])

    all_features = feature_scaler.transform(data[SELECTED_FEATURES])
    all_target = target_scaler.transform(data[[TARGET]]).ravel()
    X, y, y_dates = make_sequences(all_features, all_target, data.index, LOOKBACK)

    train_mask = y_dates < val_start
    val_mask = (y_dates >= val_start) & (y_dates < test_start)
    test_mask = y_dates >= test_start
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
        'test_dates': y_dates[test_mask],
        'target_scaler': target_scaler,
        'train_start': train.index.min(), 'train_end': train.index.max(),
        'val_start': val_start, 'val_end': pretest.index[-1],
        'test_start': test_start, 'test_end': data.index[-1],
    }

split = prepare_split(df, TEST_HORIZON)
for name in ['X_train', 'X_val', 'X_test']:
    print(name, split[name].shape)
print('Tập train:', split['train_start'].date(), 'đến', split['train_end'].date())
print('Tập validation:', split['val_start'].date(), 'đến', split['val_end'].date())
print('Tập test:', split['test_start'].date(), 'đến', split['test_end'].date())

## 6. Xây dựng và huấn luyện LSTM

Mô hình gồm hai lớp `LSTM` với lần lượt 39 và 111 units, sau đó là lớp Dense 32 units, Dropout 0.2 và một neuron đầu ra để dự báo giá. Đây là các thông số tốt nhất được paper báo cáo.

Phần tìm kiếm hyperparameter bằng Simulated Annealing của paper chưa được chạy mặc định trong demo này vì rất tốn thời gian. Ta cố định thông số tốt nhất để so sánh trực tiếp với SimpleRNN. `shuffle=False` giúp giữ nguyên thứ tự thời gian.

In [ ]:
def build_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(39, return_sequences=True),
        tf.keras.layers.LSTM(111),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(DROPOUT),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='mse',
        metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    return model

model = build_model(split['X_train'].shape[1:])
model.summary()

callbacks = []
if USE_EARLY_STOPPING:
    callbacks.append(tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True
    ))

history = model.fit(
    split['X_train'], split['y_train'],
    validation_data=(split['X_val'], split['y_val']),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1
)

## 7. Đánh giá và vẽ kết quả dự báo

In [ ]:
pred_scaled = model.predict(split['X_test'], verbose=0).ravel()
true_scaled = split['y_test']
scaler = split['target_scaler']
pred = scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
true = scaler.inverse_transform(true_scaled.reshape(-1, 1)).ravel()

metrics = {
    'model': 'LSTM',
    'test_horizon': TEST_HORIZON,
    'normalized_mse': mean_squared_error(true_scaled, pred_scaled),
    'price_mse': mean_squared_error(true, pred),
    'price_rmse': mean_squared_error(true, pred) ** 0.5,
    'price_mae': mean_absolute_error(true, pred),
}
display(pd.DataFrame([metrics]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('MSE loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(split['test_dates'], true, label='Actual')
axes[1].plot(split['test_dates'], pred, label='LSTM prediction')
axes[1].set_title(f'Copper price - last {TEST_HORIZON} observations')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Chạy thêm các horizon giống paper

Bỏ dấu `#` ở cell bên dưới nếu muốn huấn luyện riêng cho ba horizon 242, 363 và 485 phiên như paper. Cách này sẽ mất nhiều thời gian hơn vì mỗi horizon huấn luyện một mô hình mới từ đầu.

In [ ]:
HORIZONS = [242, 363, 485]
horizon_runs = []
horizon_results = []

for horizon in HORIZONS:
    print(f'\n===== Huấn luyện horizon = {horizon} =====')
    split_h = prepare_split(df, horizon)
    tf.keras.backend.clear_session()
    model_h = build_model(split_h['X_train'].shape[1:])

    callbacks_h = []
    if USE_EARLY_STOPPING:
        callbacks_h.append(tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=15, restore_best_weights=True
        ))

    history_h = model_h.fit(
        split_h['X_train'], split_h['y_train'],
        validation_data=(split_h['X_val'], split_h['y_val']),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=False,
        callbacks=callbacks_h,
        verbose=1
    )

    pred_h_scaled = model_h(split_h['X_test'], training=False).numpy().ravel()
    true_h_scaled = split_h['y_test']
    scaler_h = split_h['target_scaler']
    pred_h = scaler_h.inverse_transform(pred_h_scaled.reshape(-1, 1)).ravel()
    true_h = scaler_h.inverse_transform(true_h_scaled.reshape(-1, 1)).ravel()

    horizon_runs.append({
        'horizon': horizon,
        'dates': split_h['test_dates'],
        'true': true_h,
        'pred': pred_h,
        'history': history_h,
    })
    horizon_results.append({
        'horizon': horizon,
        'normalized_mse': mean_squared_error(true_h_scaled, pred_h_scaled),
        'price_rmse': mean_squared_error(true_h, pred_h) ** 0.5,
        'price_mae': mean_absolute_error(true_h, pred_h),
    })

horizon_results_df = pd.DataFrame(horizon_results).sort_values('horizon')
display(horizon_results_df)

In [ ]:
fig, axes = plt.subplots(
    nrows=len(horizon_runs),
    ncols=1,
    figsize=(16, 12),
    sharey=True
)

for ax, run in zip(np.atleast_1d(axes), horizon_runs):
    ax.plot(run['dates'], run['true'], label='Giá thực tế', linewidth=1.5)
    ax.plot(run['dates'], run['pred'], label='Dự đoán LSTM', linewidth=1.3)
    ax.set_title(f"LSTM - Horizon {run['horizon']} phiên")
    ax.set_ylabel('Giá đồng')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper left')
    ax.tick_params(axis='x', rotation=30)

axes[-1].set_xlabel('Thời gian')
fig.suptitle('So sánh dự báo LSTM trên 3 horizon theo paper', fontsize=15)
plt.tight_layout()
plt.show()

display(horizon_results_df)